# Flujo completo DXF -> discretizacion -> calibracion opcional

Procesa el DXF de ejemplo, exporta CSV y muestra como activar calibracion por segmento.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "dynaengine").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR = ROOT / "examples" / "data"
MATERIALS_PATH = DATA_DIR / "section_01_materials.json"
materials = json.loads(MATERIALS_PATH.read_text(encoding="utf-8"))
print(f"Proyecto: {ROOT}")
print(f"Materiales cargados: {len(materials)}")

import pandas as pd
from dynaengine import (
    CalibrationSettings,
    MaterialLibrary,
    calibrate_discretized_column,
    export_dataframe,
    process_dxf_folder,
)

Proyecto: C:\Users\joel.alarcon\Desktop\_code\prismo\external\DynaEngine
Materiales cargados: 6


In [2]:
results = process_dxf_folder(
    section_folder=DATA_DIR,
    x_positions_by_file={"section_01": {f"failure_{index}": [250.0, 480.0] for index in range(1, 8)}},
    materials=materials,
    target_frequency_hz=25,
    calibrate=False,
    failure_types_by_file={"section_01": {f"failure_{index}": "tipo_de_falla" for index in range(1, 8)}},
    small_area_scale=0.01,
)
print(f"Columnas procesadas: {len(results)}")
list(results)

Columnas procesadas: 2


['section_01-failure_3-x250p0', 'section_01-failure_7-x480p0']

In [3]:
summary = []
for name, processed in results.items():
    summary.append({
        "column_name": name,
        "raw_layers": len(processed.raw),
        "discretized_segments": len(processed.discretized),
        "max_depth_m": processed.raw["bottom_m"].max(),
        "min_frequency_hz": processed.discretized["natural_frequency_hz"].min(),
    })
summary_df = pd.DataFrame(summary)
summary_df

,column_name,raw_layers,discretized_segments,max_depth_m,min_frequency_hz
0,section_01-failure_3-x250p0,4,42,112.87278,25.328390
1,section_01-failure_7-x480p0,4,47,153.82703,25.000077


In [4]:
output_dir = ROOT / "examples" / "output"
output_dir.mkdir(parents=True, exist_ok=True)

for name, processed in results.items():
    export_dataframe(processed.raw, output_dir / f"{name}_raw.csv")
    export_dataframe(processed.discretized, output_dir / f"{name}_discretized.csv")
export_dataframe(summary_df, output_dir / "columns_summary.csv")
print(output_dir)

C:\Users\joel.alarcon\Desktop\_code\prismo\external\DynaEngine\examples\output


In [5]:
# Calibracion opcional rapida de las primeras filas discretizadas.
# Para una corrida completa, llama process_dxf_folder(..., calibrate=True).
settings = CalibrationSettings(
    optimizer="scipy",
    scipy_starts=2,
    scipy_maxiter=20,
    random_state=1,
)

first_result = next(iter(results.values()))
calibrated_preview = calibrate_discretized_column(
    first_result.discretized.head(2),
    MaterialLibrary.from_mappings(materials),
    settings=settings,
)
calibrated_preview[[
    "segment_id",
    "material_name",
    "theta_1",
    "theta_2",
    "theta_3",
    "theta_4",
    "theta_5",
    "P1",
    "P2",
    "P3",
    "dmin",
    "gqh_score",
    "mrdf_score",
]]

,segment_id,material_name,theta_1,theta_2,theta_3,theta_4,theta_5,P1,P2,P3,dmin,gqh_score,mrdf_score
0,1,Grava arcillosa,0.986060,-0.465674,0.010058,1.0,2.018916e-10,0.500000,0.525000,15.250000,5.000000,-0.059321,-1.000000e+12
1,2,Grava arcillosa,-0.418621,-0.391000,2.156224,1.0,1.737581e-01,0.599724,0.422031,2.034332,1.310093,-0.017595,-5.043542e-02
